
# 스마트팜 모델 출력값 가설 검증
**대상 파일**
- `df_active_v2.csv`
- `normality.csv`
- `ks_drift.csv`
- `hypothesis_mwu.csv`
- `hypothesis_kruskal.csv`

## 목적
이 노트북은 **오토인코더 출력값**을 기준으로, 지금 실제로 할 수 있는 검정과 할 수 없는 검정을 분리해 정리한다.  
핵심 원칙은 다음과 같다.

1. 결과를 그대로 믿지 않는다.
2. p-value 하나만으로 결론 내리지 않는다.
3. 임계값, 알람 비율, 출력 편향을 함께 본다.
4. 현재 데이터로 불가능한 검정은 억지로 돌리지 않고 `실행 불가`로 명시한다.

## 현재 파일로 가능한 항목 / 불가능한 항목
| 항목 | 현재 실행 가능 여부 | 이유 |
|---|---|---|
| 임계값 기준 점검 | 가능 | `df_active_v2.csv`에 MSE와 alarm level이 같이 있음 |
| Bootstrap 신뢰구간 | 가능 | 출력값/알람 비율에 대해 재표집 가능 |
| McNemar test | 가능(제한적) | 같은 시점에서 모델쌍의 `any alarm` 이진화 결과 비교 가능 |
| subgroup별 출력 차이 검정 | 가능(제한적) | 시점/운영조건 subgroup 기준으로 출력 편향 점검 가능 |
| paired t-test / Wilcoxon으로 모델 성능 비교 | 실행 불가 | fold별 성능표 또는 동일 라벨 기준 성능 벡터가 없음 |
| permutation test로 AUC/F1 유의성 검정 | 실행 불가 | 정답 라벨과 예측 성능지표가 없음 |
| calibration 검증 (HL / reliability) | 실행 불가 | 확률 출력과 실제 정답 라벨이 없음 |
| conformal prediction | 실행 불가 | calibration split, 잔차 구조, 목표 coverage 정의가 없음 |

## 해석 주의
- 여기서 McNemar는 **정확도 비교가 아니라 출력 비교**다.
- subgroup 검정도 **공정성의 최종 판정이 아니라 출력 편향 점검**이다.
- 정답 라벨이 없으므로, 이 노트북은 **모델 성능의 우열 판정 노트북이 아니다.**


In [ ]:

# 필요한 라이브러리를 불러옵니다.
from pathlib import Path
import itertools
import math

import numpy as np
import pandas as pd
from scipy.stats import chi2, binomtest, chi2_contingency
from IPython.display import display, Markdown

# 출력 형식을 보기 좋게 맞춥니다.
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

# 파일 경로를 현재 폴더 기준으로 찾습니다.
BASE = Path(".")
DF_PATH = BASE / "df_active_v2.csv"
NORMALITY_PATH = BASE / "normality.csv"
KS_PATH = BASE / "ks_drift.csv"
MWU_PATH = BASE / "hypothesis_mwu.csv"
KRUSKAL_PATH = BASE / "hypothesis_kruskal.csv"

# 파일을 읽습니다.
df = pd.read_csv(DF_PATH, parse_dates=["timestamp"])
normality = pd.read_csv(NORMALITY_PATH)
ks_drift = pd.read_csv(KS_PATH)
hypothesis_mwu = pd.read_csv(MWU_PATH)
hypothesis_kruskal = pd.read_csv(KRUSKAL_PATH)

# 분석 대상 AE 모델명을 정리합니다.
models = [
    "motor_current_a",
    "zone1_resistance",
    "wire_to_water_efficiency",
    "mix_ph",
]

print("로드 완료")
print(f"df_active_v2: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"normality: {normality.shape[0]:,}행")
print(f"ks_drift: {ks_drift.shape[0]:,}행")
print(f"hypothesis_mwu: {hypothesis_mwu.shape[0]:,}행")
print(f"hypothesis_kruskal: {hypothesis_kruskal.shape[0]:,}행")


로드 완료
df_active_v2: 111,893행 × 55열
normality: 13행
ks_drift: 13행
hypothesis_mwu: 17행
hypothesis_kruskal: 17행



## 1. 데이터 구조 확인
`df_active_v2.csv`에는 각 모델마다 아래 2개 컬럼이 붙어 있다.

- `mse_<모델명>`: 오토인코더 재건오차
- `alarm_<모델명>`: 임계값 기준 알람 레벨

이 노트북에서는 알람 레벨을 아래처럼 해석한다.

- `0`: Normal
- `1`: Caution
- `2`: Warning
- `3`: Error

이 구조 덕분에, **임계값이 실제로 데이터를 잘 분할하고 있는지**는 검토할 수 있다.


In [ ]:

# AE 출력 관련 컬럼만 확인합니다.
ae_cols = []
for m in models:
    ae_cols.extend([f"mse_{m}", f"alarm_{m}"])

display(df[["timestamp"] + ae_cols].head())

# 각 alarm 컬럼의 레벨 분포를 확인합니다.
alarm_counts = {}
for m in models:
    alarm_col = f"alarm_{m}"
    counts = df[alarm_col].value_counts(dropna=False).sort_index()
    alarm_counts[m] = counts.to_dict()

alarm_counts


,timestamp,mse_motor_current_a,alarm_motor_current_a,mse_zone1_resistance,alarm_zone1_resistance,mse_wire_to_water_efficiency,alarm_wire_to_water_efficiency,mse_mix_ph,alarm_mix_ph
0,2026-03-01 00:21:00,0.000775,0,0.000332,0,0.001472,0,0.036671,0
1,2026-03-01 00:22:00,0.000788,0,0.000320,0,0.001450,0,0.036804,0
2,2026-03-01 00:23:00,0.000799,0,0.000309,0,0.001428,0,0.036950,0
3,2026-03-01 00:24:00,0.000812,0,0.000299,0,0.001409,0,0.037098,0
4,2026-03-01 00:25:00,0.000823,0,0.000290,0,0.001389,0,0.037248,0


{'motor_current_a': {0: 93381, 1: 5112, 2: 8071, 3: 5329},
 'zone1_resistance': {0: 110251, 1: 103, 2: 741, 3: 798},
 'wire_to_water_efficiency': {0: 79904, 1: 3693, 2: 5404, 3: 22892},
 'mix_ph': {0: 80757, 1: 5118, 2: 25094, 3: 924}}


## 2. 모델별 알람 분포 요약
여기서는 먼저 **알람이 너무 적거나 너무 많은 모델이 없는지** 본다.  
알람 비율이 지나치게 높으면 임계값이 너무 낮을 수 있고,  
알람 비율이 지나치게 낮으면 반대로 실제 변화를 거의 못 잡고 있을 수 있다.

또한 `any alarm(>=1)`과 `severe alarm(>=2)`를 분리해서 본다.


In [ ]:

# 모델별로 단계별 개수와 비율을 계산합니다.
summary_rows = []

for m in models:
    alarm_col = f"alarm_{m}"
    mse_col = f"mse_{m}"
    vc = df[alarm_col].value_counts().sort_index()
    total = len(df)

    row = {
        "model": m,
        "n_total": total,
        "n_level0": int(vc.get(0, 0)),
        "n_level1": int(vc.get(1, 0)),
        "n_level2": int(vc.get(2, 0)),
        "n_level3": int(vc.get(3, 0)),
        "rate_any_alarm": float((df[alarm_col] >= 1).mean()),
        "rate_severe_alarm": float((df[alarm_col] >= 2).mean()),
        "mse_mean": float(df[mse_col].mean()),
        "mse_median": float(df[mse_col].median()),
    }
    summary_rows.append(row)

model_summary = pd.DataFrame(summary_rows).sort_values("rate_any_alarm", ascending=False)
display(model_summary)


,model,n_total,n_level0,n_level1,n_level2,n_level3,rate_any_alarm,rate_severe_alarm,mse_mean,mse_median
2,wire_to_water_efficiency,111893,79904,3693,5404,22892,0.285889,0.252884,0.002074,0.000532
3,mix_ph,111893,80757,5118,25094,924,0.278266,0.232526,0.033018,0.020584
0,motor_current_a,111893,93381,5112,8071,5329,0.165444,0.119757,0.005583,0.003911
1,zone1_resistance,111893,110251,103,741,798,0.014675,0.013754,0.004312,0.002456



### 1차 비판적 해석
이 표는 **좋다/나쁘다를 바로 말하는 표가 아니다.**  
대신 아래 질문을 던지기 위한 표다.

1. 한 모델만 알람 비율이 지나치게 높지 않은가?
2. severe alarm 비율이 예상보다 큰가?
3. 평균 MSE와 중앙값 MSE 차이가 너무 크지 않은가?  
   → 크다면 꼬리가 두껍거나 일부 구간에서만 점수가 급등하는 구조일 수 있다.

특히 알람 비율이 높다고 해서 무조건 좋은 것도 아니고, 나쁜 것도 아니다.  
다만 **임계값 설정을 다시 점검해야 할 후보**로 본다.



## 3. 임계값 재구성 및 분리 상태 검토
현재 CSV에는 임계값 숫자 자체가 저장돼 있지 않다.  
하지만 `mse`와 `alarm level`이 같이 있으므로, 각 레벨 사이의 경계값을 **역으로 복원**할 수 있다.

복원 방식:
- `T01`: level 0의 최대 MSE와 level 1의 최소 MSE의 중간값
- `T12`: level 1의 최대 MSE와 level 2의 최소 MSE의 중간값
- `T23`: level 2의 최대 MSE와 level 3의 최소 MSE의 중간값

같이 확인할 것:
- 레벨 간 MSE 구간이 겹치는가?
- 경계가 단조 증가하는가?
- 경계 간 틈이 너무 좁아 사실상 수치 오차 수준은 아닌가?


In [ ]:

# 레벨 사이의 경계를 역으로 복원합니다.
threshold_rows = []

for m in models:
    mse_col = f"mse_{m}"
    alarm_col = f"alarm_{m}"

    bounds = {}
    for level in [0, 1, 2, 3]:
        sub = df.loc[df[alarm_col] == level, mse_col]
        bounds[level] = {
            "min": float(sub.min()),
            "max": float(sub.max()),
            "n": int(len(sub)),
        }

    # 인접 레벨 사이의 임계값을 중간값으로 복원합니다.
    t01 = (bounds[0]["max"] + bounds[1]["min"]) / 2
    t12 = (bounds[1]["max"] + bounds[2]["min"]) / 2
    t23 = (bounds[2]["max"] + bounds[3]["min"]) / 2

    # 레벨 구간이 겹치지 않는지 확인합니다.
    no_overlap_01 = bounds[0]["max"] < bounds[1]["min"]
    no_overlap_12 = bounds[1]["max"] < bounds[2]["min"]
    no_overlap_23 = bounds[2]["max"] < bounds[3]["min"]

    threshold_rows.append({
        "model": m,
        "level0_max": bounds[0]["max"],
        "level1_min": bounds[1]["min"],
        "gap_01": bounds[1]["min"] - bounds[0]["max"],
        "T01_reconstructed": t01,
        "level1_max": bounds[1]["max"],
        "level2_min": bounds[2]["min"],
        "gap_12": bounds[2]["min"] - bounds[1]["max"],
        "T12_reconstructed": t12,
        "level2_max": bounds[2]["max"],
        "level3_min": bounds[3]["min"],
        "gap_23": bounds[3]["min"] - bounds[2]["max"],
        "T23_reconstructed": t23,
        "no_overlap_01": no_overlap_01,
        "no_overlap_12": no_overlap_12,
        "no_overlap_23": no_overlap_23,
        "strictly_increasing_thresholds": (t01 < t12 < t23),
    })

threshold_audit = pd.DataFrame(threshold_rows)
display(threshold_audit)


,model,level0_max,level1_min,gap_01,T01_reconstructed,level1_max,level2_min,gap_12,T12_reconstructed,level2_max,level3_min,gap_23,T23_reconstructed,no_overlap_01,no_overlap_12,no_overlap_23,strictly_increasing_thresholds
0,motor_current_a,0.009230,0.009230,0.000000,0.009230,0.011451,0.011453,0.000002,0.011452,0.018118,0.018121,0.000003,0.018120,True,True,True,True
1,zone1_resistance,0.014199,0.014266,0.000067,0.014232,0.017730,0.017789,0.000059,0.017760,0.028206,0.028220,0.000014,0.028213,True,True,True,True
2,wire_to_water_efficiency,0.001558,0.001559,0.000000,0.001559,0.001902,0.001902,0.000000,0.001902,0.002933,0.002933,0.000001,0.002933,True,True,True,True
3,mix_ph,0.052839,0.052841,0.000003,0.052840,0.064684,0.064687,0.000004,0.064685,0.100220,0.100237,0.000017,0.100228,True,True,True,True



### 임계값 검토 포인트
- `no_overlap_*`가 모두 `True`면, 적어도 현재 저장된 결과상 **레벨 분리는 일관적**이다.
- `gap_*`가 너무 작으면, 경계가 수치 오차나 반올림에 민감할 수 있다.
- `strictly_increasing_thresholds=True`는 **기본 sanity check 통과**로 본다.

즉, 여기서 통과해야 “임계값이 완전히 망가진 것은 아니다”라고 말할 수 있다.  
하지만 이것만으로 **좋은 threshold**라고 단정할 수는 없다.



## 4. Bootstrap 신뢰구간
정답 라벨이 없더라도, 모델 출력 자체의 불확실성은 일부 볼 수 있다.  
여기서는 아래 두 가지에 대해 bootstrap 신뢰구간을 계산한다.

- `any alarm(>=1)` 비율
- `severe alarm(>=2)` 비율

이 값이 필요한 이유:
- 재표집해도 알람 비율이 비슷하게 유지되는가?
- 모델 출력이 특정 구간에 과도하게 몰려 있지는 않은가?

주의:
- 이 CI는 **정확도 CI가 아니라 출력 비율 CI**다.


In [5]:

# bootstrap으로 알람 비율의 신뢰구간을 계산합니다.
rng = np.random.default_rng(42)
n = len(df)
n_boot = 500

bootstrap_rows = []

for m in models:
    alarm = df[f"alarm_{m}"].to_numpy()
    any_alarm = (alarm >= 1).astype(np.int8)
    severe_alarm = (alarm >= 2).astype(np.int8)

    any_rates = []
    severe_rates = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        any_rates.append(any_alarm[idx].mean())
        severe_rates.append(severe_alarm[idx].mean())

    any_ci = np.percentile(any_rates, [2.5, 50, 97.5])
    severe_ci = np.percentile(severe_rates, [2.5, 50, 97.5])

    bootstrap_rows.append({
        "model": m,
        "any_alarm_rate_obs": float(any_alarm.mean()),
        "any_alarm_ci_2.5": float(any_ci[0]),
        "any_alarm_ci_50": float(any_ci[1]),
        "any_alarm_ci_97.5": float(any_ci[2]),
        "severe_alarm_rate_obs": float(severe_alarm.mean()),
        "severe_alarm_ci_2.5": float(severe_ci[0]),
        "severe_alarm_ci_50": float(severe_ci[1]),
        "severe_alarm_ci_97.5": float(severe_ci[2]),
    })

bootstrap_summary = pd.DataFrame(bootstrap_rows).sort_values("any_alarm_rate_obs", ascending=False)
display(bootstrap_summary)


,model,any_alarm_rate_obs,any_alarm_ci_2.5,any_alarm_ci_50,any_alarm_ci_97.5,severe_alarm_rate_obs,severe_alarm_ci_2.5,severe_alarm_ci_50,severe_alarm_ci_97.5
2,wire_to_water_efficiency,0.285889,0.283029,0.286055,0.288584,0.252884,0.250225,0.253045,0.255625
3,mix_ph,0.278266,0.275600,0.278337,0.280815,0.232526,0.230059,0.232615,0.234939
0,motor_current_a,0.165444,0.163356,0.165390,0.167783,0.119757,0.117920,0.119766,0.121603
1,zone1_resistance,0.014675,0.013919,0.014643,0.015355,0.013754,0.013021,0.013732,0.014385



### Bootstrap 해석
- CI 폭이 매우 좁으면, 출력 비율 자체는 표본 재추출에 대해 안정적이다.
- 그러나 **안정적 = 타당함**은 아니다.  
  예를 들어 임계값이 지나치게 낮아 알람이 과도하게 많이 나오는 구조도, 표본 수가 크면 매우 안정적으로 반복된다.

따라서 이 결과는 **재현성 점검**이지, **정답성 인증**이 아니다.



## 5. McNemar test로 모델쌍 출력 비교
사용자 요청 목록에는 McNemar test가 포함돼 있다.  
현재 파일로 가능한 방식은 다음뿐이다.

- 각 모델의 `alarm level >= 1`을 `any_alarm` 이진값으로 바꾼다.
- 같은 timestamp에서 모델 A와 B가 서로 다른 알람을 냈는지 비교한다.
- McNemar test로 **두 모델의 양성 판정 성향이 같은지** 본다.

중요:
- 이것은 **정확도 비교가 아니다.**
- 정답 라벨이 없으므로, “어느 모델이 더 잘 맞는다”는 결론은 금지다.
- 여기서는 **출력 경향의 차이**만 본다.


In [6]:

# McNemar exact test를 직접 계산하는 함수입니다.
def mcnemar_exact_from_binary(y1, y2):
    y1 = np.asarray(y1).astype(int)
    y2 = np.asarray(y2).astype(int)

    n00 = int(((y1 == 0) & (y2 == 0)).sum())
    n01 = int(((y1 == 0) & (y2 == 1)).sum())
    n10 = int(((y1 == 1) & (y2 == 0)).sum())
    n11 = int(((y1 == 1) & (y2 == 1)).sum())

    discordant = n01 + n10

    if discordant == 0:
        p_exact = 1.0
        chi_stat_cc = 0.0
        p_chi_cc = 1.0
    else:
        # continuity correction이 들어간 근사 카이제곱값입니다.
        chi_stat_cc = (abs(n01 - n10) - 1) ** 2 / discordant
        p_chi_cc = 1 - chi2.cdf(chi_stat_cc, df=1)

        # exact binomial p-value입니다.
        p_exact = binomtest(min(n01, n10), n=discordant, p=0.5, alternative="two-sided").pvalue

    agreement = (n00 + n11) / len(y1)

    if n10 > n01:
        direction = "model_a_more_positive"
    elif n01 > n10:
        direction = "model_b_more_positive"
    else:
        direction = "balanced"

    return {
        "n00": n00,
        "n01": n01,
        "n10": n10,
        "n11": n11,
        "discordant_total": discordant,
        "discordant_rate": discordant / len(y1),
        "agreement_rate": agreement,
        "mcnemar_chi2_cc": chi_stat_cc,
        "p_value_exact": p_exact,
        "p_value_chi_cc": p_chi_cc,
        "direction": direction,
    }

pair_rows = []

for a, b in itertools.combinations(models, 2):
    y_a = (df[f"alarm_{a}"] >= 1).astype(int)
    y_b = (df[f"alarm_{b}"] >= 1).astype(int)

    stat = mcnemar_exact_from_binary(y_a, y_b)
    stat["model_a"] = a
    stat["model_b"] = b
    pair_rows.append(stat)

mcnemar_df = pd.DataFrame(pair_rows)[[
    "model_a", "model_b",
    "n00", "n01", "n10", "n11",
    "discordant_total", "discordant_rate", "agreement_rate",
    "p_value_exact", "p_value_chi_cc", "direction"
]].sort_values(["p_value_exact", "discordant_rate"])

display(mcnemar_df)


,model_a,model_b,n00,n01,n10,n11,discordant_total,discordant_rate,agreement_rate,p_value_exact,p_value_chi_cc,direction
2,motor_current_a,mix_ph,79897,13484,860,17652,14344,0.128194,0.871806,0.000000,0.000000,model_b_more_positive
1,motor_current_a,wire_to_water_efficiency,78791,14590,1113,17399,15703,0.140339,0.859661,0.000000,0.000000,model_b_more_positive
0,motor_current_a,zone1_resistance,92956,425,17295,1217,17720,0.158366,0.841634,0.000000,0.000000,model_a_more_positive
3,zone1_resistance,wire_to_water_efficiency,79359,30892,545,1097,31437,0.280956,0.719044,0.000000,0.000000,model_b_more_positive
4,zone1_resistance,mix_ph,79645,30606,1112,530,31718,0.283467,0.716533,0.000000,0.000000,model_b_more_positive
5,wire_to_water_efficiency,mix_ph,76137,3767,4620,27369,8387,0.074956,0.925044,0.000000,0.000000,model_a_more_positive



### McNemar 해석 규칙
- `p_value_exact`가 매우 작더라도, 표본 수가 매우 크면 작은 차이도 유의하게 나올 수 있다.
- 따라서 반드시 `discordant_rate`, `agreement_rate`, `direction`을 같이 본다.
- 이 표에서 중요한 질문은:
  1. 두 모델이 실제로 많이 다르게 울리나?
  2. 어느 쪽이 더 자주 양성 알람을 내나?
  3. 그 차이가 운영상 무시 가능한 수준인가?

즉, McNemar는 여기서 **“모델 A와 B가 같은 신호를 내는가”**를 보는 도구다.



## 6. subgroup별 출력 편향 점검
공정성/편향 항목은 보통 protected attribute가 있을 때 수행한다.  
하지만 현재 파일에는 그런 정보가 없다.  
대신 운영 맥락에서 최소한의 **출력 편향 audit**은 가능하다.

여기서는 예시로 `day / night` subgroup를 만든다.
- `06:00 ~ 17:59` → day
- 그 외 → night

이후 각 모델의 `any_alarm` 비율이 주간/야간에 유의하게 다른지 본다.


In [7]:

# day / night subgroup를 만듭니다.
df["hour"] = df["timestamp"].dt.hour
df["day_night"] = np.where(df["hour"].between(6, 17), "day", "night")

subgroup_rows = []

for m in models:
    y = (df[f"alarm_{m}"] >= 1).astype(int)
    table = pd.crosstab(df["day_night"], y)

    # 2x2 교차표에 대한 카이제곱 검정입니다.
    chi2_stat, p_value, _, _ = chi2_contingency(table)

    # Cramer's V는 표본 수가 매우 클 때 p-value만 보는 문제를 줄이기 위한 보조 효과크기입니다.
    n_total = table.to_numpy().sum()
    cramers_v = np.sqrt(chi2_stat / (n_total * min(table.shape[0] - 1, table.shape[1] - 1)))

    day_rate = table.loc["day", 1] / table.loc["day"].sum()
    night_rate = table.loc["night", 1] / table.loc["night"].sum()

    subgroup_rows.append({
        "model": m,
        "day_any_alarm_rate": float(day_rate),
        "night_any_alarm_rate": float(night_rate),
        "night_minus_day": float(night_rate - day_rate),
        "chi2_p_value": float(p_value),
        "cramers_v": float(cramers_v),
        "day_n": int(table.loc["day"].sum()),
        "night_n": int(table.loc["night"].sum()),
    })

subgroup_audit = pd.DataFrame(subgroup_rows).sort_values("cramers_v", ascending=False)
display(subgroup_audit)


,model,day_any_alarm_rate,night_any_alarm_rate,night_minus_day,chi2_p_value,cramers_v,day_n,night_n
0,motor_current_a,0.084336,0.277048,0.192711,0.000000,0.256021,64800,47093
1,zone1_resistance,0.008549,0.023103,0.014554,0.000000,0.059679,64800,47093
2,wire_to_water_efficiency,0.301188,0.264838,-0.036351,0.000000,0.039698,64800,47093
3,mix_ph,0.267438,0.293165,0.025726,0.000000,0.028321,64800,47093



### subgroup audit 해석
- 여기서 유의하다고 해서 바로 “편향 모델”이라고 단정하면 안 된다.
- 하지만 주간/야간 차이가 지나치게 크면, 아래를 의심할 수 있다.
  - threshold가 특정 운영조건에 편향됨
  - 입력 스케일이나 분포가 낮/밤에 달라짐
  - 모델이 구조적으로 특정 구간을 더 민감하게 울림

즉 이 표는 **배포 전 경고등**으로 쓰는 것이 맞다.



## 7. 기존 결과 파일을 현재 노트북에 참고자료로 연결
업로드된 CSV에는 이미 원시/출력 관련 검정 결과가 일부 저장돼 있다.  
다만 이 파일들은 **모델 성능 검정**이 아니라,  
주로 **분포/드리프트/알람 단계 차이**를 보여주는 참고자료다.

여기서는 상위 몇 개만 빠르게 확인한다.


In [8]:

# 기존 결과 파일에서 상위 항목을 일부만 확인합니다.
display(Markdown("### normality.csv 상위 10개"))
display(normality.head(10))

display(Markdown("### ks_drift.csv 상위 10개"))
display(ks_drift.head(10))

display(Markdown("### hypothesis_mwu.csv 상위 10개"))
display(hypothesis_mwu.head(10))

display(Markdown("### hypothesis_kruskal.csv 상위 10개"))
display(hypothesis_kruskal.head(10))


### normality.csv 상위 10개

,컬럼,왜도,SW p-value,KS p-value,정규분포?,권장 검정
0,wire_to_water_efficiency,31.490000,0.000000,0.000000,X,Mann-Whitney 권장
1,zone1_resistance,14.380000,0.000000,0.000000,X,Mann-Whitney 권장
2,mix_temp_c,0.560000,0.000000,0.000000,X,Mann-Whitney 권장
3,pump_rpm,-0.320000,0.000000,0.000000,X,Mann-Whitney 권장
4,dosing_acid_ml_min,-0.310000,0.000000,0.000000,X,Mann-Whitney 권장
5,discharge_pressure_kpa,-0.240000,0.000000,0.000000,X,Mann-Whitney 권장
6,suction_pressure_kpa,0.200000,0.000000,0.000000,X,Mann-Whitney 권장
7,mix_flow_l_min,0.170000,0.000000,0.000000,X,Mann-Whitney 권장
8,motor_current_a,-0.130000,0.000000,0.000000,X,Mann-Whitney 권장
9,acid_tank_level_pct,0.130000,0.000000,0.000000,X,Mann-Whitney 권장


### ks_drift.csv 상위 10개

,컬럼,KS statistic,p-value,drift 감지,전반 중앙값,후반 중앙값,변화율(%)
0,acid_tank_level_pct,0.979100,0.000000,O (주의),69.725500,45.365000,-34.900000
1,bearing_temperature_c,0.833100,0.000000,O (주의),40.677000,44.750000,10.000000
2,motor_temperature_c,0.765600,0.000000,O (주의),43.083000,47.747000,10.800000
3,zone1_resistance,0.553100,0.000000,O (주의),27.850900,54.518400,95.800000
4,motor_current_a,0.520600,0.000000,O (주의),5.532000,7.839000,41.700000
5,discharge_pressure_kpa,0.519700,0.000000,O (주의),173.548500,214.545000,23.600000
6,suction_pressure_kpa,0.491500,0.000000,O (주의),-10.158000,-13.073000,-28.700000
7,wire_to_water_efficiency,0.468800,0.000000,O (주의),0.038000,0.021500,-43.300000
8,mix_flow_l_min,0.447400,0.000000,O (주의),24.433500,13.885000,-43.200000
9,mix_temp_c,0.423600,0.000000,O (주의),19.015000,18.368000,-3.400000


### hypothesis_mwu.csv 상위 10개

,AE 모델,변수,정상 중앙값,이상 중앙값,변화율(%),MWU p,t-test p,KS p,효과크기 r,Bonferroni 기준,유의(MWU)
0,motor_current_a,motor_current_a,6.267000,0.085000,-98.600000,0.000000,비정규,0.000000,0.547000,0.012500,V
1,motor_current_a,motor_temperature_c,45.150000,47.659000,5.600000,0.000000,비정규,0.000000,0.460000,0.012500,V
2,motor_current_a,pump_rpm,1726.831000,1.037000,-99.900000,0.000000,비정규,0.000000,0.570000,0.012500,V
3,motor_current_a,discharge_pressure_kpa,186.578000,3.053000,-98.400000,0.000000,비정규,0.000000,0.546000,0.012500,V
4,zone1_resistance,zone1_resistance,49.085400,86.304600,75.800000,0.000000,비정규,0.000000,0.304000,0.016670,V
5,zone1_resistance,zone1_pressure_kpa,130.893000,56.738000,-56.700000,0.000000,비정규,0.000000,0.302000,0.016670,V
6,zone1_resistance,zone1_flow_l_min,3.425000,0.991000,-71.100000,0.000000,비정규,0.000000,0.408000,0.016670,V
7,wire_to_water_efficiency,wire_to_water_efficiency,0.029000,0.021300,-26.700000,0.000000,비정규,0.000000,0.243000,0.010000,V
8,wire_to_water_efficiency,pump_rpm,1720.586000,1707.054000,-0.800000,0.000000,비정규,0.000000,0.111000,0.010000,V
9,wire_to_water_efficiency,motor_temperature_c,44.485000,48.665000,9.400000,0.000000,비정규,0.000000,0.830000,0.010000,V


### hypothesis_kruskal.csv 상위 10개

,AE 모델,변수,KW statistic,p-value,그룹간 차이,레벨0(정상),레벨1(주의),레벨2(경고),레벨3(위험)
0,motor_current_a,motor_current_a,17398.830000,0.000000,V,6.142000,9.200000,0.083000,0.088000
1,motor_current_a,motor_temperature_c,19819.500000,0.000000,V,44.936000,50.543000,47.576000,47.825000
2,motor_current_a,pump_rpm,12491.940000,0.000000,V,1724.022000,1755.970000,0.977000,1.145000
3,motor_current_a,discharge_pressure_kpa,17380.950000,0.000000,V,184.399000,240.337500,3.042000,3.071000
4,zone1_resistance,zone1_resistance,960.810000,0.000000,V,49.098600,31.033100,52.148100,1875500.000000
5,zone1_resistance,zone1_pressure_kpa,522.680000,0.000000,V,130.921000,116.683000,81.846000,2.140500
6,zone1_resistance,zone1_flow_l_min,1171.370000,0.000000,V,3.425000,3.682000,1.551000,0.000000
7,wire_to_water_efficiency,wire_to_water_efficiency,4428.160000,0.000000,V,0.029400,0.021800,0.021700,0.013900
8,wire_to_water_efficiency,pump_rpm,1566.190000,0.000000,V,1721.321500,1710.403000,1710.580500,1187.434500
9,wire_to_water_efficiency,motor_temperature_c,48039.740000,0.000000,V,44.334000,48.422000,48.798500,48.322000



## 8. 현재 데이터로는 불가능한 검정
아래 항목은 **지금 업로드된 파일만으로는 실행하면 안 된다.**

### 8-1. paired t-test / Wilcoxon으로 모델 성능 비교
불가 이유:
- fold별 성능표가 없다.
- 모델별 동일 기준 성능 벡터(AUC/F1/MAE/RMSE)가 없다.
- 현재 있는 것은 **행별 출력값**이지 **반복 평가 성능표**가 아니다.

### 8-2. permutation test로 AUC/F1 유의성 검정
불가 이유:
- 정답 라벨이 없다.
- random baseline과 비교할 분류 성능지표가 없다.

### 8-3. calibration 검증 (Hosmer-Lemeshow, reliability diagram)
불가 이유:
- 확률 예측값이 없다.
- 실제 발생 여부 라벨이 없다.

### 8-4. conformal prediction
불가 이유:
- calibration split이 없다.
- coverage target이 없다.
- 잔차 기반/분위수 기반 설계를 할 정보가 없다.

즉 이 노트북은 **모델 출력 audit**까지는 가능하지만,  
**모델 성능 검증 최종본**이 되려면 정답 라벨 또는 비교용 성능표가 추가로 필요하다.



## 9. 최종 점검표
아래 표는 “통과 / 재검토 / 실행 불가”만 명확히 남기기 위한 표다.


In [9]:

# 최종 점검표를 만듭니다.
check_rows = []

# 1) 임계값 단조성 / 겹침 여부
for _, row in threshold_audit.iterrows():
    passed = bool(
        row["no_overlap_01"] and
        row["no_overlap_12"] and
        row["no_overlap_23"] and
        row["strictly_increasing_thresholds"]
    )
    check_rows.append({
        "section": "threshold_sanity",
        "model": row["model"],
        "status": "PASS" if passed else "REVIEW",
        "reason": "레벨 간 MSE 구간이 겹치지 않고 경계가 단조 증가함" if passed else "레벨 구간 중첩 또는 경계 역전이 있음",
    })

# 2) 알람 비율 경고
for _, row in model_summary.iterrows():
    rate = row["rate_any_alarm"]
    if rate >= 0.50:
        status = "REVIEW"
        reason = "any alarm 비율이 50% 이상이라 threshold가 공격적일 수 있음"
    elif rate <= 0.02:
        status = "REVIEW"
        reason = "any alarm 비율이 매우 낮아 실제 변화 포착력이 낮을 수 있음"
    else:
        status = "PASS"
        reason = "알람 비율이 극단적이지 않음"
    check_rows.append({
        "section": "alarm_prevalence",
        "model": row["model"],
        "status": status,
        "reason": reason,
    })

# 3) subgroup 편차 경고
for _, row in subgroup_audit.iterrows():
    # 매우 큰 표본에서는 p-value가 거의 항상 작게 나올 수 있으므로
    # 여기서는 effect size(Cramer's V)를 함께 봅니다.
    if row["cramers_v"] >= 0.10:
        status = "REVIEW"
        reason = "day/night 출력 차이가 작지 않음"
    else:
        status = "PASS"
        reason = "day/night 출력 차이가 아주 크지는 않음"
    check_rows.append({
        "section": "subgroup_output_disparity",
        "model": row["model"],
        "status": status,
        "reason": reason,
    })

# 4) 현재 데이터로 불가능한 항목
for section_name, why in [
    ("paired_metric_comparison", "fold별 성능표 또는 동일 기준 평가 벡터가 없음"),
    ("permutation_auc_f1", "정답 라벨과 성능지표가 없음"),
    ("calibration", "확률 출력과 실제 라벨이 없음"),
    ("conformal_prediction", "calibration split과 coverage 정의가 없음"),
]:
    check_rows.append({
        "section": section_name,
        "model": "[ALL]",
        "status": "UNAVAILABLE",
        "reason": why,
    })

final_checklist = pd.DataFrame(check_rows)
display(final_checklist)


,section,model,status,reason
0,threshold_sanity,motor_current_a,PASS,레벨 간 MSE 구간이 겹치지 않고 경계가 단조 증가함
1,threshold_sanity,zone1_resistance,PASS,레벨 간 MSE 구간이 겹치지 않고 경계가 단조 증가함
2,threshold_sanity,wire_to_water_efficiency,PASS,레벨 간 MSE 구간이 겹치지 않고 경계가 단조 증가함
3,threshold_sanity,mix_ph,PASS,레벨 간 MSE 구간이 겹치지 않고 경계가 단조 증가함
4,alarm_prevalence,wire_to_water_efficiency,PASS,알람 비율이 극단적이지 않음
5,alarm_prevalence,mix_ph,PASS,알람 비율이 극단적이지 않음
6,alarm_prevalence,motor_current_a,PASS,알람 비율이 극단적이지 않음
7,alarm_prevalence,zone1_resistance,REVIEW,any alarm 비율이 매우 낮아 실제 변화 포착력이 낮을 수 있음
8,subgroup_output_disparity,motor_current_a,REVIEW,day/night 출력 차이가 작지 않음
9,subgroup_output_disparity,zone1_resistance,PASS,day/night 출력 차이가 아주 크지는 않음



## 10. 결론 메모
이 노트북 기준으로 지금 말할 수 있는 핵심은 아래와 같다.

1. **임계값 분리 자체는 일관적인지**  
   → `threshold_sanity`에서 확인

2. **알람 비율이 과도하지 않은지**  
   → `alarm_prevalence`에서 확인

3. **모델쌍이 같은 시점에 전혀 다른 신호를 내는지**  
   → `McNemar` 결과에서 확인

4. **특정 운영조건(day/night)에 과민한지**  
   → `subgroup_output_disparity`에서 확인

5. **정답 라벨이 없어서 아직 못 하는 검정은 무엇인지**  
   → `UNAVAILABLE`로 명시

즉, 이 노트북은 **모델 출력값 audit 노트북**으로는 충분히 의미가 있지만,  
**모델 배포 최종 승인 노트북**은 아니다.  
배포 최종판까지 가려면 아래 중 최소 하나가 더 필요하다.

- 정답 라벨
- fold별 성능표
- 확률 출력값
- calibration 전용 split
